# structure of agent reply


In [29]:
from pydantic import BaseModel
from typing import List

class Task(BaseModel):
    id: int
    name: str
    description: str
    agent_type: str    
    url: str 

In [30]:
class TaskPlan(BaseModel):
    user_requirement: str
    tasks: List[Task]

## API keys

In [31]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [32]:
from google import genai

gemini_client = genai.Client(
    api_key=GOOGLE_API_KEY
)

print("Gemini client created")

Gemini client created


In [5]:
user_requirement = """
Find me the best laptop under ₹80,000
for programming, gaming and college use.
"""

In [7]:
orchestrator_prompt = f"""
You are the Orchestrator of a shopping research system.

Your job is to understand the user's requirement
and break it into research tasks.

You are NOT doing the research.

For each task determine:

- task id
- task name
- what the task should accomplish
- which type of agent should perform it

Available agent types:

browser
llm
logic

Use browser when information must be collected from websites.
Use llm for reasoning, summarization or analysis.
Use logic for deterministic calculations.

User requirement:

{user_requirement}
"""

## Orchestrator generator

In [8]:
response = gemini_client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=orchestrator_prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": TaskPlan.model_json_schema(),
    },
)

In [9]:
print(response.text)

{
  "user_requirement": "Find me the best laptop under ₹80,000 for programming, gaming and college use.",
  "tasks": [
    {
      "id": 1,
      "name": "Search Laptops",
      "description": "Search e-commerce websites and tech retail stores for laptops priced under ₹80,000 suitable for programming, gaming, and college students.",
      "agent_type": "browser",
      "url": "https://www.amazon.in"
    },
    {
      "id": 2,
      "name": "Extract Laptop Specs and Prices",
      "description": "Extract key specifications such as CPU, GPU, RAM, storage, battery life, and exact pricing for the shortlisted laptops from search results.",
      "agent_type": "browser",
      "url": "https://www.flipkart.com"
    },
    {
      "id": 3,
      "name": "Analyze and Compare Laptops",
      "description": "Analyze the gathered specifications against the requirements for programming performance, gaming capability, and college portability to rank the top choices.",
      "agent_type": "llm",
   

## convert to python objs, not json
## also maintains the TaskPlan Schema


In [10]:
task_plan = TaskPlan.model_validate_json(response.text)

In [11]:
print(task_plan)

user_requirement='Find me the best laptop under ₹80,000 for programming, gaming and college use.' tasks=[Task(id=1, name='Search Laptops', description='Search e-commerce websites and tech retail stores for laptops priced under ₹80,000 suitable for programming, gaming, and college students.', agent_type='browser', url='https://www.amazon.in'), Task(id=2, name='Extract Laptop Specs and Prices', description='Extract key specifications such as CPU, GPU, RAM, storage, battery life, and exact pricing for the shortlisted laptops from search results.', agent_type='browser', url='https://www.flipkart.com'), Task(id=3, name='Analyze and Compare Laptops', description='Analyze the gathered specifications against the requirements for programming performance, gaming capability, and college portability to rank the top choices.', agent_type='llm', url=''), Task(id=4, name='Calculate Value Scores', description='Perform deterministic calculations to score the laptops based on price-to-performance ratio,

In [12]:
import pprint
# type(task_plan)
pprint.pprint(task_plan)

TaskPlan(user_requirement='Find me the best laptop under ₹80,000 for programming, gaming and college use.', tasks=[Task(id=1, name='Search Laptops', description='Search e-commerce websites and tech retail stores for laptops priced under ₹80,000 suitable for programming, gaming, and college students.', agent_type='browser', url='https://www.amazon.in'), Task(id=2, name='Extract Laptop Specs and Prices', description='Extract key specifications such as CPU, GPU, RAM, storage, battery life, and exact pricing for the shortlisted laptops from search results.', agent_type='browser', url='https://www.flipkart.com'), Task(id=3, name='Analyze and Compare Laptops', description='Analyze the gathered specifications against the requirements for programming performance, gaming capability, and college portability to rank the top choices.', agent_type='llm', url=''), Task(id=4, name='Calculate Value Scores', description='Perform deterministic calculations to score the laptops based on price-to-performa

In [38]:
for task in task_plan.tasks:
    print(
        task.id,
        "|",
        task.name,
        "|",
        task.agent_type
    )

1 | Market Search | browser
2 | Feature Analysis | llm
3 | Score Calculation | logic
4 | Final Recommendation | llm


## same thing with OPENAI

In [13]:
from openai import OpenAI

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)

print("OpenAI client created")

OpenAI client created


In [14]:
response_openai = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": """
You are the Orchestrator of a shopping research system.

Your job is to understand a user's requirement
and break it into research tasks.

You are NOT doing the research.

Available agent types:

- browser
- llm
- logic

Use browser for website research.
Use llm for reasoning and summarization.
Use logic for deterministic calculations.
"""
        },
        {
            "role": "user",
            "content": user_requirement
        }
    ],
    text_format=TaskPlan,
)

In [15]:
task_plan_openai = response_openai.output_parsed

In [16]:
print(task_plan_openai)

user_requirement='Find the best laptop under ₹80,000 suitable for programming, gaming, and college use.' tasks=[Task(id=1, name='Research laptops for programming and gaming', description='Look for laptops under ₹80,000 that excel in both programming and gaming performance.', agent_type='browser', url='https://www.flipkart.com/search?q=laptop+under+80000+for+programming+and+gaming'), Task(id=2, name='Summarize laptop features', description='Summarize the key features, specifications, and pros/cons of the top choices identified in the browser research.', agent_type='llm', url=''), Task(id=3, name='Calculate performance metrics', description='Determine the best price-to-performance ratio for the identified laptops, considering CPU, GPU, and RAM.', agent_type='logic', url='')]


In [17]:
print("===== GEMINI =====")

for task in task_plan.tasks:
    print(
        task.id,
        task.name,
        "→",
        task.agent_type
    )

print("\n===== OPENAI =====")

for task in task_plan_openai.tasks:
    print(
        task.id,
        task.name,
        "→",
        task.agent_type
    )

===== GEMINI =====
1 Search Laptops → browser
2 Extract Laptop Specs and Prices → browser
3 Analyze and Compare Laptops → llm
4 Calculate Value Scores → logic

===== OPENAI =====
1 Research laptops for programming and gaming → browser
2 Summarize laptop features → llm
3 Calculate performance metrics → logic


In [33]:
user_requirement = """
Find the best laptop under ₹80,000 for programming,
gaming and college use.
Compare products from Amazon and Flipkart.
"""

## Return the response in `TaskPlan` format.

In [34]:
def create_plan_gemini(user_requirement: str) -> TaskPlan:
    prompt = f"""
You are the Orchestrator of a shopping research system.

Your job is to understand the user's requirement
and break it into research tasks.

You are NOT doing the research.

For each task determine:

- task id
- task name
- what the task should accomplish
- which type of agent should perform it
- URL to visit if it is a browser task

Available agent types:

browser
llm
logic

Use browser when information must be collected from websites.
Use llm for reasoning, summarization or analysis.
Use logic for deterministic calculations.

For browser tasks:
- provide the exact website URL to visit
- create separate tasks when research should be performed on different websites
- make the task description specific about what information needs to be collected

User requirement:

{user_requirement}
"""
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": TaskPlan.model_json_schema(),
        },
    )
    return TaskPlan.model_validate_json(response.text)

In [35]:
plan = create_plan_gemini(
    "Find the best laptop under ₹80,000 for programming and gaming."
)



#### input

In [21]:
print(plan.user_requirement)

Find the best laptop under ₹80,000 for programming and gaming.


#### output of Orchestrator

In [36]:
for task in plan.tasks:
    print("ID:", task.id)
    print("NAME:", task.name)
    print("TYPE:", task.agent_type)
    print("URL:", task.url)
    print("DESCRIPTION:", task.description)
    print("-" * 50)
  

ID: 1
NAME: Search Amazon India for Gaming Laptops
TYPE: browser
URL: https://www.amazon.in/s?k=gaming+laptop+under+80000
DESCRIPTION: Search for laptops under ₹80,000 suitable for programming and gaming, and extract product names, specifications, and prices.
--------------------------------------------------
ID: 2
NAME: Search Flipkart for Gaming Laptops
TYPE: browser
URL: https://www.flipkart.com/search?q=gaming%20laptop%20under%2080000
DESCRIPTION: Search for laptops under ₹80,000 suitable for programming and gaming, and extract product names, specifications, and prices.
--------------------------------------------------
ID: 3
NAME: Analyze and Compare Laptop Specifications
TYPE: llm
URL: 
DESCRIPTION: Analyze the collected laptop options from Amazon and Flipkart, evaluating processors, RAM, GPU, and reviews to recommend the best programming and gaming laptop under ₹80,000.
--------------------------------------------------


In [37]:
import subprocess

## fetch only those result where  `agent_type=browser`

In [24]:
import json

browser_task=[]

for task in plan.tasks:
    if task.agent_type == "browser":
        browser_task.append(task.model_dump())
        
browser_task

[{'id': 1,
  'name': 'Search Amazon for gaming laptops',
  'description': 'Search for laptops priced under 80000 INR suitable for programming and gaming. Collect model names, specifications (CPU, GPU, RAM, Storage), and prices.',
  'agent_type': 'browser',
  'url': 'https://www.amazon.in'},
 {'id': 2,
  'name': 'Search Flipkart for gaming laptops',
  'description': 'Search for laptops priced under 80000 INR suitable for programming and gaming. Collect model names, specifications (CPU, GPU, RAM, Storage), and prices.',
  'agent_type': 'browser',
  'url': 'https://www.flipkart.com'}]

that desciption  of finding laptops is sent in to the browser use

### creating the first_browser agent `(Amazon)`

In [30]:
result_amazon = subprocess.run(
    [
        "python",
        "../agents/product_discovery.py",
        browser_task[0]['url'],
        browser_task[0]["description"]
    ],
    capture_output=True,
    text=True,
    encoding="utf-8"
)

In [33]:
print("return code:", result_amazon.returncode)
print("STDOUT:", result_amazon.stdout)
print("STDERR:", result_amazon.stderr)


return code: 0
STDOUT: {
  "products": [
    {
      "name": "ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6\"(39.6 cm), Windows 11 Home, Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop",
      "price": 73490.0,
      "rating": 4.3,
      "specifications": "AMD Ryzen 7, RTX 3050-4GB, 16GB RAM, 512GB SSD, 15.6\" FHD",
      "url": null,
      "source": "Amazon.in"
    },
    {
      "name": "Acer ALG, Intel Core5-210H Processor, NVIDIA GeForceRTX 3050-4GB DDR6, 16GB RAM/ 512GB SSD, FHD 15.6\", 144Hz, Win 11 Home, Titanium Gray, 1.99 KG, AL15G-53, Gaming Laptop",
      "price": 76999.0,
      "rating": 3.0,
      "specifications": "Intel Core 5-210H, NVIDIA GeForce RTX 3050 4GB, 16GB RAM, 512GB SSD, 15.6\" FHD 144Hz",
      "url": null,
      "source": "Amazon.in"
    },
    {
      "name": "MSI Thin 15, Intel 13th Gen. Core i5-13420H, 40CM FHD 144Hz Gaming Laptop(16GB/512GB NVMe SSD/Windows 11 Home/NVIDIA GeForce RTX 2050,GDDR6 4GB /Cosm

now convert it to dict

In [32]:
amazon_products=json.loads(result_amazon.stdout)["products"]
print(amazon_products)

[{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm), Windows 11 Home, Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop', 'price': 73490.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7, RTX 3050-4GB, 16GB RAM, 512GB SSD, 15.6" FHD', 'url': None, 'source': 'Amazon.in'}, {'name': 'Acer ALG, Intel Core5-210H Processor, NVIDIA GeForceRTX 3050-4GB DDR6, 16GB RAM/ 512GB SSD, FHD 15.6", 144Hz, Win 11 Home, Titanium Gray, 1.99 KG, AL15G-53, Gaming Laptop', 'price': 76999.0, 'rating': 3.0, 'specifications': 'Intel Core 5-210H, NVIDIA GeForce RTX 3050 4GB, 16GB RAM, 512GB SSD, 15.6" FHD 144Hz', 'url': None, 'source': 'Amazon.in'}, {'name': 'MSI Thin 15, Intel 13th Gen. Core i5-13420H, 40CM FHD 144Hz Gaming Laptop(16GB/512GB NVMe SSD/Windows 11 Home/NVIDIA GeForce RTX 2050,GDDR6 4GB /Cosmos Gray/1.86Kg), B13UCX-1807IN', 'price': 79900.0, 'rating': 4.2, 'specifications': 'Intel 13th Gen Core i5-13420H, NVIDIA GeForce RTX 2050 4GB, 16

### creating the second_browser agent `(Flipkart)`

In [34]:
result_flipkart = subprocess.run(
    [
        "python",
        "../agents/product_discovery.py",
        browser_task[1]['url'],
        browser_task[1]["description"]
    ],
    capture_output=True,
    text=True,
    encoding="utf-8"
)

In [35]:
flipkart_products=json.loads(result_flipkart.stdout)["products"]
print(flipkart_products)



[{'name': 'Acer Aspire 7 (i5 14th Gen) Intel Core 5 210H - (16 GB/512 GB SSD/Windows 11 Home/6 GB Graphics/NVIDIA', 'price': 75990.0, 'rating': 4.3, 'specifications': 'Intel Core 5 Processor, 16 GB DDR4 RAM, 512 GB SSD, 6 GB Graphics (NVIDIA), Windows 11 Home', 'url': 'https://www.flipkart.com/search?q=gaming%20laptop%20under%2080000', 'source': 'Flipkart'}, {'name': 'Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H - (16 GB/512 GB SSD/Windows 11 Home/6 GB Graphics/NVIDIA', 'price': 79990.0, 'rating': 3.9, 'specifications': 'Intel Core 7 Processor, 16 GB DDR4 RAM, 512 GB SSD, 6 GB Graphics (NVIDIA), Windows 11 Home', 'url': 'https://www.flipkart.com/search?q=gaming%20laptop%20under%2080000', 'source': 'Flipkart'}, {'name': 'MSI Thin 15 Intel Core i7 13th Gen 13620H - (16 GB/512 GB SSD/Windows 11 Home/4 GB Graphics/NVIDIA GeForce', 'price': 79990.0, 'rating': 4.5, 'specifications': 'Intel Core i7 Processor (13th Gen), 16 GB DDR4 RAM, 512 GB SSD, 4 GB Graphics (NVIDIA), Windows 11 Home', 'u

In [238]:
total_products=amazon_products+flipkart_products
print(total_products)

[{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop', 'price': 70990.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB SSD, FHD 15.6", Windows 11 Home', 'url': 'https://www.amazon.in/ASUS-FA506NCQ-HN006W-Gaming-Laptop/dp/B0F4J1C1V4/', 'source': 'Amazon India'}, {'name': 'ASUS TUF A15 (2025), Smartchoice, AMD Ryzen 7 7445HS, Gaming Laptop, RTX 3050-4GB, 75W TGP, 16GB RAM (Upgradeable Upto 64GB) 1TB SSD, FHD, 15.6", 144Hz, M365 Basic(1Y), Office 2024, Black, 2.3 Kg, FA506NCG-HN251WS', 'price': 77990.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7 7445HS, RTX 3050-4GB, 16GB RAM, 1TB SSD, FHD 15.6", 144Hz, Windows 11', 'url': 'https://www.amazon.in/ASUS-FA506NCG-HN251WS-Gaming-Laptop/dp/B0F4J1C1V5/', 'source': 'Amazon India'}, {'name': 'Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H - (16 GB/512 GB SSD/Windows 11 Home/6 GB Graph

### convert that dict to dataframe

In [239]:
import pandas as pd

df = pd.DataFrame(total_products)

df

,name,price,rating,specifications,url,source
0,"ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 1...",70990.0,4.3,"AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB...",https://www.amazon.in/ASUS-FA506NCQ-HN006W-Gam...,Amazon India
1,"ASUS TUF A15 (2025), Smartchoice, AMD Ryzen 7 ...",77990.0,4.3,"AMD Ryzen 7 7445HS, RTX 3050-4GB, 16GB RAM, 1T...",https://www.amazon.in/ASUS-FA506NCG-HN251WS-Ga...,Amazon India
2,Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H ...,75990.0,3.9,"Intel Core 7 Processor, 16 GB DDR4 RAM, 64 bit...",https://www.flipkart.com/search?q=laptop+under...,Flipkart
3,DELL G15 Intel Core i5 13th Gen 13450HX - (16 ...,79990.0,4.2,"Intel Core i5 Processor (13th Gen), 16 GB DDR5...",https://www.flipkart.com/search?q=laptop+under...,Flipkart
4,Acer Aspire 7 (i5 14th Gen) Intel Core 5 210H ...,77990.0,4.3,"Intel Core 5 Processor, 16 GB DDR4 RAM, 64 bit...",https://www.flipkart.com/search?q=laptop+under...,Flipkart
5,HP Victus Intel Core i5 13th Gen 13420H - (16 ...,77990.0,4.4,"Intel Core i5 Processor (13th Gen), 16 GB DDR4...",https://www.flipkart.com/search?q=laptop+under...,Flipkart


In [166]:
import json

with open(
    "../outputs/amazon_products.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        products,
        f,
        indent=2,
        ensure_ascii=False
    )

## Enhancing the Product schema

In [ ]:
# from typing import List,Dict

# class Product_gen(BaseModel):
#     name:str
#     price:float|None=None
#     rating:float|None=None
#     brand:str|None=None
#     url:str|None=None
#     source:str|None=None
#     specifications:Dict[str,str]={}

#### better architecture with automatic browser agents

In [38]:
import json

browser_task=[]

for task in plan.tasks:
    if task.agent_type == "browser":
        browser_task.append(task.model_dump())
        
browser_task

[{'id': 1,
  'name': 'Search Amazon India for Gaming Laptops',
  'description': 'Search for laptops under ₹80,000 suitable for programming and gaming, and extract product names, specifications, and prices.',
  'agent_type': 'browser',
  'url': 'https://www.amazon.in/s?k=gaming+laptop+under+80000'},
 {'id': 2,
  'name': 'Search Flipkart for Gaming Laptops',
  'description': 'Search for laptops under ₹80,000 suitable for programming and gaming, and extract product names, specifications, and prices.',
  'agent_type': 'browser',
  'url': 'https://www.flipkart.com/search?q=gaming%20laptop%20under%2080000'}]

In [39]:
all_products = []

for task in browser_task:

    result = subprocess.run(
        [
            "python",
            "../agents/product_discovery.py",
            task["url"],
            task["description"]
        ],
        capture_output=True,
        text=True,
        encoding="utf-8"
    )

    # parse result
    products=json.loads(result.stdout)["products"]
    # add products
    all_products.extend(products)
print(all_products)    

[{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop', 'price': 73490.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6" (39.6 cm), Windows 11 Home, Graphite Black, 2.3 Kg', 'url': 'https://www.amazon.in/ASUS-3050-4GB-Upgradeable-Graphite-FA506NCQ-HN006W/dp/B0GW85JTBN/ref=sr_1_3', 'source': 'Amazon India'}, {'name': "HP Victus, 13th Gen Intel Core i5-13420H, 4GB RTX 2050, 16GB DDR4 (Upgradeable) 512GB SSD, 144Hz, IPS, 300 nits, FHD, 15.6''/39.6cm, Win11, M365* Office24, Mica Silver, 2.3kg, fa2703tx Gaming Laptop", 'price': 72990.0, 'rating': 3.8, 'specifications': '13th Gen Intel Core i5-13420H, 4GB RTX 2050, 16GB DDR4 (Upgradeable), 512GB SSD, 144Hz, IPS, 300 nits, FHD, 15.6"/39.6cm, Win11, M365/Office24, Mica Silver, 2.3kg', 'url': 'https://www.amazon.in/HP-i5-13420H-Upgrade-Office24-fa2703tx/dp

In [38]:
import pandas as pd

df = pd.DataFrame(all_products)

df

,name,price,rating,specifications,url,source
0,"ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 1...",73490.0,4.3,"AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB...",https://www.amazon.in/dp/D059BCFE,Amazon.in
1,"HP Victus, 13th Gen Intel Core i5-13420H, 4GB ...",72990.0,3.8,"13th Gen Intel Core i5-13420H, 4GB RTX 2050, 1...",https://www.amazon.in/dp/A47C02FB,Amazon.in
2,Lenovo V15 G4 AMD Ryzen 5 7520U 15.6 inch FHD ...,54890.0,4.0,"AMD Ryzen 5 7520U, AMD Graphics, 16GB DDR5 RAM...",https://www.amazon.in/dp/AE48C4ED,Amazon.in
3,"Acer ALG, Intel Core5-210H Processor, NVIDIA G...",76999.0,3.0,"Intel Core 5-210H, NVIDIA GeForce RTX 3050 4GB...",https://www.amazon.in/dp/E7427C82,Amazon.in
4,Acer Aspire 7(i5 14th Gen) Intel Core 5 210H -...,71990.0,4.5,"Intel Core 5 Processor, 16 GB DDR4 RAM, 64 bit...",https://www.flipkart.com,Flipkart


In [43]:
print("Total products:", len(all_products))

Total products: 5


In [44]:
names = [product["name"] for product in all_products]

print("Total:", len(names))
print("Unique names:", len(set(names)))

Total: 5
Unique names: 5


# let's begin Engineered Normalization.

In [45]:
import re

def normalize_name(name):
    name = name.lower()

    # remove punctuation
    name = re.sub(r"[^a-z0-9\s]", " ", name)

    # remove extra spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name

In [46]:
test = all_products[0]["name"]

print("Original:")
print(test)

print("\nNormalized:")
print(normalize_name(test))

Original:
ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop

Normalized:
asus tuf a15 amd ryzen 7 170 rtx 3050 4gb 16gb ram upgradeable 512gb ssd fhd 15 6 39 6 cm windows 11 home graphite black 2 3 kg fa506ncq hn006w gaming laptop


### inserted normalized name in each products

In [47]:
for product in all_products:
    product["normalized_name"] = normalize_name(product["name"])

In [48]:
all_products[0]

{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop',
 'price': 73490.0,
 'rating': 4.3,
 'specifications': 'AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB SSD, FHD 15.6" Display, Windows 11 Home',
 'url': 'https://www.amazon.in/dp/D059BCFE',
 'source': 'Amazon.in',
 'normalized_name': 'asus tuf a15 amd ryzen 7 170 rtx 3050 4gb 16gb ram upgradeable 512gb ssd fhd 15 6 39 6 cm windows 11 home graphite black 2 3 kg fa506ncq hn006w gaming laptop'}

### Data cleansing, finding same normalized names.
### A certain level of cleansing we are performiing, and after that we will send it to gemini to find duplicate prods.

In [49]:
from collections import Counter

name_counts = Counter(
    product["normalized_name"]
    for product in all_products
)

In [50]:
duplicates = {
    name: count
    for name, count in name_counts.items()
    if count > 1
}

duplicates

{}

In [53]:
def get_candidate_key(name):
    normalized = normalize_name(name)
    words = normalized.split()

    return " ".join(words[:10])

In [54]:
for product in all_products:
    print(
        get_candidate_key(product["name"]),
        "|",
        product["source"]
    )

asus tuf a15 amd ryzen 7 170 rtx 3050 4gb | Amazon.in
hp victus 13th gen intel core i5 13420h 4gb rtx | Amazon.in
lenovo v15 g4 amd ryzen 5 7520u 15 6 inch | Amazon.in
acer alg intel core5 210h processor nvidia geforcertx 3050 4gb | Amazon.in
acer aspire 7 i5 14th gen intel core 5 210h | Flipkart


In [55]:
from collections import defaultdict
groups = defaultdict(list)

for product in all_products:
    key = get_candidate_key(product["name"])
    groups[key].append(product)

In [56]:
for key, products in groups.items():

    print("\nGROUP:", key)

    for product in products:
        print(
            " ",
            product["source"],
            "|",
            product["name"],
            "| ₹",
            product["price"]
        )


GROUP: asus tuf a15 amd ryzen 7 170 rtx 3050 4gb
  Amazon.in | ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop | ₹ 73490.0

GROUP: hp victus 13th gen intel core i5 13420h 4gb rtx
  Amazon.in | HP Victus, 13th Gen Intel Core i5-13420H, 4GB RTX 2050, 16GB DDR4 (Upgradeable) 512GB SSD, 144Hz, IPS, 300 nits, FHD, 15.6''/39.6cm, Win11, M365* Office24, Mica Silver, 2.3kg, fa2703tx Gaming Laptop | ₹ 72990.0

GROUP: lenovo v15 g4 amd ryzen 5 7520u 15 6 inch
  Amazon.in | Lenovo V15 G4 AMD Ryzen 5 7520U 15.6 inch FHD Laptop, AMD Graphics, 16GB DDR5 5500Mhz Ram, 512GB SSD NVMe, Windows 11, Dolby Audio, Arctic Grey, 1 Year Onsite Brand Warranty | ₹ 54890.0

GROUP: acer alg intel core5 210h processor nvidia geforcertx 3050 4gb
  Amazon.in | Acer ALG, Intel Core5-210H Processor, NVIDIA GeForceRTX 3050-4GB DDR6,16GB RAM/ 512GB SSD, FHD 15.6", 144Hz, Win 11 Home, Titanium Gray, 

## LLM as a judge for same product

In [57]:
from pydantic import BaseModel


class MatchResult(BaseModel):
    same_product: bool
    confidence: float
    reason: str

In [58]:
def match_products(product1, product2):

    prompt = f"""
You are a product matching system.

Determine whether the following two listings
refer to the SAME physical product.

Product 1:
Name: {product1["name"]}
Specifications: {product1.get("specifications")}
Source: {product1["source"]}

Product 2:
Name: {product2["name"]}
Specifications: {product2.get("specifications")}
Source: {product2["source"]}

Important:
- Ignore differences in seller/source.
- Ignore price differences.
- Ignore minor formatting differences.
- Look for model numbers, product identifiers,
  specifications and other identifying information.
- Do not assume two products are the same just
  because their names are similar.

Return:
- same_product: true or false
- confidence: number between 0 and 1
- reason: short explanation
"""
    
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": MatchResult.model_json_schema(),
        },
    )

    return MatchResult.model_validate_json(response.text)

In [59]:
for i, product in enumerate(all_products):
    print(i, product["name"], "|", product["source"])

0 ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop | Amazon.in
1 HP Victus, 13th Gen Intel Core i5-13420H, 4GB RTX 2050, 16GB DDR4 (Upgradeable) 512GB SSD, 144Hz, IPS, 300 nits, FHD, 15.6''/39.6cm, Win11, M365* Office24, Mica Silver, 2.3kg, fa2703tx Gaming Laptop | Amazon.in
2 Lenovo V15 G4 AMD Ryzen 5 7520U 15.6 inch FHD Laptop, AMD Graphics, 16GB DDR5 5500Mhz Ram, 512GB SSD NVMe, Windows 11, Dolby Audio, Arctic Grey, 1 Year Onsite Brand Warranty | Amazon.in
3 Acer ALG, Intel Core5-210H Processor, NVIDIA GeForceRTX 3050-4GB DDR6,16GB RAM/ 512GB SSD, FHD 15.6", 144Hz, Win 11 Home, Titanium Gray, 1.99 KG, AL15G-53, Gaming Laptop | Amazon.in
4 Acer Aspire 7(i5 14th Gen) Intel Core 5 210H - (16 GB/512 GB SSD/Windows 11 Home/6 GB Graphics/NVIDIA ... | Flipkart


In [60]:
product1 = all_products[0]
product2 = all_products[1]
product1

{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop',
 'price': 73490.0,
 'rating': 4.3,
 'specifications': 'AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB SSD, FHD 15.6" Display, Windows 11 Home',
 'url': 'https://www.amazon.in/dp/D059BCFE',
 'source': 'Amazon.in',
 'normalized_name': 'asus tuf a15 amd ryzen 7 170 rtx 3050 4gb 16gb ram upgradeable 512gb ssd fhd 15 6 39 6 cm windows 11 home graphite black 2 3 kg fa506ncq hn006w gaming laptop'}

In [61]:
match_result = match_products(product1, product2)

print(match_result)

same_product=False confidence=1.0 reason='The products are from completely different brands and series: Product 1 is an ASUS TUF A15 with an AMD processor and RTX 3050, while Product 2 is an HP Victus with an Intel processor and RTX 2050.'


## Deduplicate products across sources (LLM-assisted)

Apply `match_products` only *within* each candidate-key group (cheap, since blocking already
narrowed candidates), merge transitively-matched listings with a small union-find, then
collapse each cluster into one canonical product record.

In [62]:
def dedupe_group(products, threshold=0.7):
    """Cluster duplicate listings within a single candidate-key group."""
    n = len(products)

    # 0 or 1 listing: nothing to compare, no LLM call needed.
    if n <= 1:
        return [products] if n == 1 else []

    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[rj] = ri

    # Only pairs *within* this (already blocked) group are judged by the LLM.
    for i in range(n):
        for j in range(i + 1, n):
            result = match_products(products[i], products[j])
            if result.same_product and result.confidence >= threshold:
                union(i, j)

    clusters = defaultdict(list)
    for i in range(n):
        clusters[find(i)].append(products[i])

    return list(clusters.values())

In [40]:
def merge_cluster(cluster):
    """Collapse a cluster of duplicate listings into one canonical product."""

    # Prefer the listing with the most detailed specs as the canonical record.
    canonical = max(cluster, key=lambda p: len(p.get("specifications") or ""))

    listings = [
        {
            "source": p.get("source"),
            "price": p.get("price"),
            "rating": p.get("rating"),
            "url": p.get("url"),
        }
        for p in cluster
    ]

    prices = [l["price"] for l in listings if l["price"] is not None]

    return {
        "name": canonical["name"],
        "normalized_name": canonical.get("normalized_name"),
        "specifications": canonical.get("specifications"),
        "listings": listings,
        "sources": [l["source"] for l in listings],
        "min_price": min(prices) if prices else None,
    }

In [63]:
unique_products = []

for key, group_products in groups.items():
    for cluster in dedupe_group(group_products):
        unique_products.append(merge_cluster(cluster))

print("Total listings:", len(all_products))
print("Unique products:", len(unique_products))

Total listings: 5
Unique products: 5


In [64]:
df_unique = pd.DataFrame(unique_products)
df_unique

,name,normalized_name,specifications,listings,sources,min_price
0,"ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 1...",asus tuf a15 amd ryzen 7 170 rtx 3050 4gb 16gb...,"AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB...","[{'source': 'Amazon.in', 'price': 73490.0, 'ra...",[Amazon.in],73490.0
1,"HP Victus, 13th Gen Intel Core i5-13420H, 4GB ...",hp victus 13th gen intel core i5 13420h 4gb rt...,"13th Gen Intel Core i5-13420H, 4GB RTX 2050, 1...","[{'source': 'Amazon.in', 'price': 72990.0, 'ra...",[Amazon.in],72990.0
2,Lenovo V15 G4 AMD Ryzen 5 7520U 15.6 inch FHD ...,lenovo v15 g4 amd ryzen 5 7520u 15 6 inch fhd ...,"AMD Ryzen 5 7520U, AMD Graphics, 16GB DDR5 RAM...","[{'source': 'Amazon.in', 'price': 54890.0, 'ra...",[Amazon.in],54890.0
3,"Acer ALG, Intel Core5-210H Processor, NVIDIA G...",acer alg intel core5 210h processor nvidia gef...,"Intel Core 5-210H, NVIDIA GeForce RTX 3050 4GB...","[{'source': 'Amazon.in', 'price': 76999.0, 'ra...",[Amazon.in],76999.0
4,Acer Aspire 7(i5 14th Gen) Intel Core 5 210H -...,acer aspire 7 i5 14th gen intel core 5 210h 16...,"Intel Core 5 Processor, 16 GB DDR4 RAM, 64 bit...","[{'source': 'Flipkart', 'price': 71990.0, 'rat...",[Flipkart],71990.0


In [65]:
with open(
    "../outputs/unique_products.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        unique_products,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved", len(unique_products), "unique products to ../outputs/unique_products.json")

Saved 5 unique products to ../outputs/unique_products.json


# Research Agent


In [66]:
class ProductResearch(BaseModel):
    product_name: str
    specifications: dict[str, str]
    source_urls: list[str]

In [67]:
product = unique_products[0]

print("PRODUCT:")
print(product["name"])

print("\nLISTINGS:")
for listing in product["listings"]:
    print(listing)

PRODUCT:
ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop

LISTINGS:
{'source': 'Amazon.in', 'price': 73490.0, 'rating': 4.3, 'url': 'https://www.amazon.in/dp/D059BCFE'}


In [68]:
product = unique_products[0]

print(product["name"])


ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM (Upgradeable), 512GB SSD, FHD, 15.6"(39.6 cm),Windows 11 Home,Graphite Black, 2.3 Kg, FA506NCQ-HN006W, Gaming Laptop


In [69]:
for listing in product["listings"]:
    print(listing["source"])
    print(listing["url"])
    print()

Amazon.in
https://www.amazon.in/dp/D059BCFE



In [70]:
listing = product["listings"][0]

In [74]:
url = "https://www.amazon.in"
product_name = product["name"]
url

'https://www.amazon.in'

# YOUTUBE RESEARCH

In [4]:
from googleapiclient.discovery import build
from dotenv import load_dotenv
import os

load_dotenv(override=True)

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

youtube = build(
    "youtube", 
    "v3", 
    developerKey=YOUTUBE_API_KEY
    )



### Search for the top 20 results on Youtube, and get their id's

In [22]:
def search_videos(query,max_no_results=20):

    request=youtube.search().list(
        q=query,
        part="snippet",
        type="video",
        maxResults=max_no_results,
        order="relevance"
    )

    response=request.execute()
    
    videos = []
    for item in response["items"]:
        video_meta_data={
        "video_id":item["id"]["videoId"],
        "title":item["snippet"]["title"],
        "thumbnail_url":item["snippet"]["thumbnails"]["default"]["url"],
        "published": item["snippet"]["publishedAt"],
        "description": item["snippet"]["description"],
        "channel":item["snippet"]["channelTitle"]
        }
        videos.append(video_meta_data)


    return videos



In [23]:
search_videos("Sony WH-1000XM5 review")

[{'video_id': 'aWWHU8-5oqU',
  'title': 'Sony WH 1000XM5 Review - Audiophile Perspective | The Best ANC Headphones?',
  'thumbnail_url': 'https://i.ytimg.com/vi/aWWHU8-5oqU/default.jpg',
  'published': '2022-11-06T06:30:00Z',
  'description': "Sony's 2022 iteration of its popular XM series of headphones is the new WH 1000XM5. I've used this for a while now and here's ...",
  'channel': 'Trakin Tech English'},
 {'video_id': '6CsJZxfZsL0',
  'title': 'Sony WH-1000XM5 Review: Two Steps Forward, One Step Back!',
  'thumbnail_url': 'https://i.ytimg.com/vi/6CsJZxfZsL0/default.jpg',
  'published': '2022-05-12T16:01:05Z',
  'description': "Sony's MK5 noise cancelling headphones are still king of the hill, Sony WH1000XM5: https://geni.us/F6x4cC Sony WH1000XM4: ...",
  'channel': 'Marques Brownlee'},
 {'video_id': 'qnEKEI4ZoHo',
  'title': 'Sony WH-1000XM5 Review - Taylor Swift Sounds Good in These',
  'thumbnail_url': 'https://i.ytimg.com/vi/qnEKEI4ZoHo/default.jpg',
  'published': '2022-06-16T

### Fetch some stats of a video eg:-views,likes,total commnets

In [6]:
def get_video_stats(video_id):
    request=youtube.videos().list(
        part="statistics",
        id=video_id
    )
    response=request.execute()
    stats = response["items"][0]["statistics"]
    video_stats={
        "views": int(stats.get("viewCount", 0)),
        "likes": int(stats.get("likeCount", 0)),
        "comment_count": int(stats.get("commentCount", 0))
    }
    return video_stats


In [24]:
get_video_stats("k_fbRCo-yAs")

{'views': 87651, 'likes': 3521, 'comment_count': 300}

### Find top commnets of a single video

In [7]:
def get_top_comments(video_id, max_results=30):
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        order="relevance",   # gets highest-engagement comments first
        maxResults=max_results,
        textFormat="plainText"
    )
    response = request.execute()
    comments = []
    for item in response["items"]:
        top = item["snippet"]["topLevelComment"]["snippet"]
        comments.append({
            "text": top["textDisplay"],
            "likes": top["likeCount"],
            "author": top["authorDisplayName"]
        })
    return comments

In [25]:
get_top_comments("k_fbRCo-yAs")

[{'text': 'We need that kind of mic test in every video going forward',
  'likes': 14,
  'author': '@Ok_Sounds_Good'},
 {'text': '2:35 "jabbaaa tera ghar bar kidar hai" 😂',
  'likes': 3,
  'author': '@devil-d67'},
 {'text': 'Alreay using these from Last 3 months. Best headphone ever used❤. Beast of ANC🔥',
  'likes': 11,
  'author': '@TechEncoders'},
 {'text': '7:37 KR dollar sign. Bhai apna dollar ka fan 🎉',
  'likes': 1,
  'author': '@caabhisheksingh6300'},
 {'text': "I'm watching this while wearing these headphones. The KING of noise cancellation.❤",
  'likes': 12,
  'author': '@benmetalhead'},
 {'text': 'I loving watching tech reviews from Venom\'s Tech. So I thought of watching the review of a product I already own and I am not disappointed. I am a dialysis patient and I spend 4 hours in dialysis thrice a week, there is too much noise in the dialysis center. So whenever there is too much sound, I turn on the noise-cancelling on this headphone and enjoy the peace "me time". Thank yo

## The `importan` part..is to get the Transcript of a video!!!!

In [8]:
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()

def get_transcript(video_id):
    try:
        fetched_transcript = ytt_api.fetch(video_id)
        # fetched_transcript is iterable of snippets with .text, .start, .duration
        return " ".join(snippet.text for snippet in fetched_transcript)
    except Exception as e:
        print(f"Transcript error for {video_id}: {type(e).__name__}: {e}")
        return None

In [28]:
get_transcript("9a-3r2llDjI")

"so it's been one month since sony launched their new flagship headphones to 1000 x mark 5's and when they came out there was a lot of excitement a lot of claims that these are the best of the best the best wireless headphones you can buy but now that it's been a month i want to talk about what it's actually like to use these as i've been using them in many situations on a day-to-day basis from working out to commuting to working in an office to just hanging out and listening to music and all of those i found some things that are not just based on the spec sheet so i want to spend this video talking a little bit more about the nuances of these headphones to help you decide whether or not they're actually the best pair for you to buy so this is sort of like a day in the life i kind of want to break this down into different categories of like when i was using them so what it's like to use them when you're working out what it's like to use them when you're commuting and i want to break it

# ALL IN ONE PLACE
1. `search_product`
2. `get some basic infos`
3. `get comments`
4. `get trnascripts`

In [19]:
query = "Sony WH-1000XM5 review"

videos = search_videos(query, max_no_results=5)

for video in videos:

    print("\n" + "=" * 70)

    print("TITLE:", video["title"])
    print("CHANNEL:", video["channel"])
    print("VIDEO ID:", video["video_id"])

    # Get statistics
    stats = get_video_stats(video["video_id"])

    print("\nSTATS:")
    print("Views:", stats["views"])
    print("Likes:", stats["likes"])
    print("Comments:", stats["comment_count"])

    # Get comments
    comments = get_top_comments(
        video["video_id"],
        max_results=10
    )

    print("\nTOP COMMENTS:")

    for comment in comments:
        print(
            f'{comment["likes"]} likes - '
            f'{comment["author"]}: '
            f'{comment["text"]}'
        )

    # Get transcript
    transcript = get_transcript(video["video_id"])

    print("\nTRANSCRIPT:")

    if transcript:
        print(transcript[:2000])
    else:
        print("Transcript unavailable")


TITLE: Sony WH 1000XM5 Review - Audiophile Perspective | The Best ANC Headphones?
CHANNEL: Trakin Tech English
VIDEO ID: aWWHU8-5oqU

STATS:
Views: 50651
Likes: 1355
Comments: 109

TOP COMMENTS:
1 likes - @tgfan972: waiting for video on "best over/on ear Headphones in every budget "
1 likes - @shilpasankpal7350: Yes I recently purchased it 
It’s voice cancellation feature is amazing . Adaptive voice cancellation is very useful . Comfort of headphones is amazing . I got it for 26k from I world shop
0 likes - @ashunegi6216: bro iam convinced that you have great music taste you listening to TOOL's undertow sober is mind blowing
3 likes - @vasqora: Can you do a over the ear headphone list for under 10k for beginner audiophiles. i basically needed something with a mic and above average audio quality. i checked out reviews for some gaming headphones as well but cant bring my self to trust them, wireless would be appreciated.
3 likes - @abhijitbagchi3471: Please make a playlist, your song re